In [2]:
# DM4ML -Assignment - Validation

import json
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd

# ============================================================
# RecoMart Validation - robust file discovery + validation
# ============================================================

# ====== SET THIS FIRST ======
# Example:
# PROJECT_ROOT = Path(r"C:\Users\barath\dm4ml_recomart\src\ingestion\recomart_pipeline\data")
#PROJECT_ROOT = Path(r"C:\Users\barath\dm4ml_recomart\src\ingestion\recomart_pipeline").resolve()
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
#DATA_ROOT = PROJECT_ROOT / "data"

RAW_ROOT = PROJECT_ROOT / "data" / "raw"
BRONZE_ROOT = PROJECT_ROOT / "data" / "bronze"
REPORT_ROOT = PROJECT_ROOT / "reports" / "validation"
LOG_ROOT = PROJECT_ROOT / "logs"

REPORT_ROOT.mkdir(parents=True, exist_ok=True)
LOG_ROOT.mkdir(parents=True, exist_ok=True)

UTC_NOW = datetime.now(timezone.utc)
RUN_ID = UTC_NOW.strftime("%Y%m%dT%H%M%SZ")
RUN_TS = UTC_NOW.isoformat()

LOG_FILE = LOG_ROOT / f"validation_log_{RUN_ID}.jsonl"
ISSUES_FILE = REPORT_ROOT / f"validation_issues_{RUN_ID}.csv"
SUMMARY_FILE = REPORT_ROOT / f"validation_summary_{RUN_ID}.csv"
REPORT_FILE = REPORT_ROOT / f"data_quality_report_{RUN_ID}.json"

ALLOWED_EVENT_TYPES = {"view", "addtocart", "transaction"}

def log_event(stage, status, message, extra=None):
    record = {
        "event_ts": datetime.now(timezone.utc).isoformat(),
        "stage": stage,
        "status": status,
        "message": message,
        "run_id": RUN_ID,
    }
    if extra:
        record.update(extra)
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps(record) + "\n")

def add_issue(issues, dataset, check_type, severity, issue_count, description, file_path=None):
    issues.append({
        "dataset": dataset,
        "check_type": check_type,
        "severity": severity,
        "issue_count": int(issue_count),
        "description": description,
        "file_path": str(file_path) if file_path else None,
    })

def latest_match(base_dir: Path, pattern: str):
    if not base_dir.exists():
        return None
    matches = list(base_dir.rglob(pattern))
    if not matches:
        return None
    return max(matches, key=lambda p: p.stat().st_mtime)

def missing_columns(df, expected_columns):
    return [c for c in expected_columns if c not in df.columns]

def duplicate_count(df, subset=None):
    return int(df.duplicated(subset=subset).sum())

def null_summary(df):
    result = {}
    for c in df.columns:
        n = int(df[c].isna().sum())
        if n > 0:
            result[c] = n
    return result

def invalid_numeric_range_count(df, column, min_value=None, max_value=None):
    if column not in df.columns:
        return None
    s = pd.to_numeric(df[column], errors="coerce")
    mask = pd.Series(False, index=df.index)
    if min_value is not None:
        mask = mask | (s < min_value)
    if max_value is not None:
        mask = mask | (s > max_value)
    return int(mask.fillna(False).sum())

def invalid_membership_count(df, column, allowed_values):
    if column not in df.columns:
        return None
    return int((~df[column].isin(allowed_values)).sum())

def invalid_timestamp_count(df, column, unit=None):
    if column not in df.columns:
        return None
    parsed = pd.to_datetime(df[column], errors="coerce", unit=unit)
    return int(parsed.isna().sum())

def build_dataset_summary(dataset_name, df, issues):
    dataset_issues = [x for x in issues if x["dataset"] == dataset_name]
    error_count = sum(x["issue_count"] for x in dataset_issues if x["severity"] == "error")
    warning_count = sum(x["issue_count"] for x in dataset_issues if x["severity"] == "warning")
    return {
        "dataset": dataset_name,
        "rows": int(len(df)) if df is not None else 0,
        "columns": int(len(df.columns)) if df is not None else 0,
        "errors": int(error_count),
        "warnings": int(warning_count),
        "status": "PASS" if error_count == 0 else "FAIL",
    }

def discover_files():
    return {
        "events_csv": latest_match(RAW_ROOT, "events.csv"),
        "category_tree_csv": latest_match(RAW_ROOT, "category_tree.csv"),
        "item_properties_part1_csv": latest_match(RAW_ROOT, "item_properties_part1.csv"),
        "item_properties_part2_csv": latest_match(RAW_ROOT, "item_properties_part2.csv"),
        "products_raw_json": latest_match(RAW_ROOT, "products_raw.json"),
        "categories_raw_json": latest_match(RAW_ROOT, "categories_raw.json"),
        "products_parquet": latest_match(BRONZE_ROOT, "products.parquet"),
        "categories_parquet": latest_match(BRONZE_ROOT, "categories.parquet"),
    }

def print_discovered_files(files):
    print("PROJECT_ROOT:", PROJECT_ROOT)
    print("RAW_ROOT:", RAW_ROOT)
    print("BRONZE_ROOT:", BRONZE_ROOT)
    print("\nDiscovered files:")
    for k, v in files.items():
        print(f"- {k}: {v}")

def validate_file_presence(files, issues):
    required = [
        "events_csv",
        "category_tree_csv",
        "item_properties_part1_csv",
        "item_properties_part2_csv",
        "products_raw_json",
        "categories_raw_json",
    ]
    for key in required:
        if files.get(key) is None:
            add_issue(
                issues,
                dataset=key,
                check_type="file_presence",
                severity="error",
                issue_count=1,
                description=f"Required file not found for {key}. Check PROJECT_ROOT or run ingestion first.",
            )

def validate_retailrocket(files, issues, summaries):
    if not all([
        files["events_csv"],
        files["category_tree_csv"],
        files["item_properties_part1_csv"],
        files["item_properties_part2_csv"],
    ]):
        return

    events = pd.read_csv(files["events_csv"])
    category_tree = pd.read_csv(files["category_tree_csv"])
    item_properties_1 = pd.read_csv(files["item_properties_part1_csv"])
    item_properties_2 = pd.read_csv(files["item_properties_part2_csv"])
    item_properties = pd.concat([item_properties_1, item_properties_2], ignore_index=True)

    exp_events = ["timestamp", "visitorid", "event", "itemid", "transactionid"]
    exp_category = ["categoryid", "parentid"]
    exp_item_props = ["timestamp", "itemid", "property", "value"]

    mc = missing_columns(events, exp_events)
    if mc:
        add_issue(issues, "retailrocket_events", "schema", "error", len(mc), f"Missing columns: {mc}", files["events_csv"])
    for col, cnt in null_summary(events).items():
        add_issue(issues, "retailrocket_events", "missing_values", "warning", cnt, f"Nulls in {col}", files["events_csv"])
    dups = duplicate_count(events, subset=["timestamp", "visitorid", "event", "itemid"])
    if dups > 0:
        add_issue(issues, "retailrocket_events", "duplicates", "warning", dups, "Duplicate rows", files["events_csv"])
    bad_event = invalid_membership_count(events, "event", ALLOWED_EVENT_TYPES)
    if bad_event and bad_event > 0:
        add_issue(issues, "retailrocket_events", "domain", "error", bad_event, "Invalid event values", files["events_csv"])
    bad_ts = invalid_timestamp_count(events, "timestamp", unit="ms")
    if bad_ts and bad_ts > 0:
        add_issue(issues, "retailrocket_events", "format", "error", bad_ts, "Invalid timestamps", files["events_csv"])
    summaries.append(build_dataset_summary("retailrocket_events", events, issues))

    mc = missing_columns(category_tree, exp_category)
    if mc:
        add_issue(issues, "retailrocket_category_tree", "schema", "error", len(mc), f"Missing columns: {mc}", files["category_tree_csv"])
    for col, cnt in null_summary(category_tree).items():
        add_issue(issues, "retailrocket_category_tree", "missing_values", "warning", cnt, f"Nulls in {col}", files["category_tree_csv"])
    dups = duplicate_count(category_tree, subset=["categoryid", "parentid"])
    if dups > 0:
        add_issue(issues, "retailrocket_category_tree", "duplicates", "warning", dups, "Duplicate rows", files["category_tree_csv"])
    summaries.append(build_dataset_summary("retailrocket_category_tree", category_tree, issues))

    mc = missing_columns(item_properties, exp_item_props)
    if mc:
        add_issue(issues, "retailrocket_item_properties", "schema", "error", len(mc), f"Missing columns: {mc}", str(files["item_properties_part1_csv"]))
    for col, cnt in null_summary(item_properties).items():
        add_issue(issues, "retailrocket_item_properties", "missing_values", "warning", cnt, f"Nulls in {col}", str(files["item_properties_part1_csv"]))
    dups = duplicate_count(item_properties, subset=["timestamp", "itemid", "property", "value"])
    if dups > 0:
        add_issue(issues, "retailrocket_item_properties", "duplicates", "warning", dups, "Duplicate rows", str(files["item_properties_part1_csv"]))
    bad_ts = invalid_timestamp_count(item_properties, "timestamp", unit="ms")
    if bad_ts and bad_ts > 0:
        add_issue(issues, "retailrocket_item_properties", "format", "error", bad_ts, "Invalid timestamps", str(files["item_properties_part1_csv"]))
    summaries.append(build_dataset_summary("retailrocket_item_properties", item_properties, issues))

def validate_dummyjson(files, issues, summaries):
    if not all([files["products_raw_json"], files["categories_raw_json"]]):
        return

    with open(files["products_raw_json"], "r", encoding="utf-8") as f:
        products_payload = json.load(f)

    with open(files["categories_raw_json"], "r", encoding="utf-8") as f:
        categories_payload = json.load(f)

    if "products" not in products_payload:
        add_issue(issues, "dummyjson_products_raw", "schema", "error", 1, "Missing 'products' key", files["products_raw_json"])
        products_df = pd.DataFrame()
    else:
        products_df = pd.json_normalize(products_payload["products"], sep="_")

    if isinstance(categories_payload, list):
        if len(categories_payload) > 0 and isinstance(categories_payload[0], dict):
            categories_df = pd.json_normalize(categories_payload, sep="_")
        else:
            categories_df = pd.DataFrame({"category": categories_payload})
    else:
        categories_df = pd.DataFrame()
        add_issue(issues, "dummyjson_categories_raw", "schema", "error", 1, "categories_raw.json must be a list", files["categories_raw_json"])

    exp_products = ["id", "title", "category", "price"]
    mc = missing_columns(products_df, exp_products)
    if mc:
        add_issue(issues, "dummyjson_products_raw", "schema", "error", len(mc), f"Missing columns: {mc}", files["products_raw_json"])
    for col, cnt in null_summary(products_df).items():
        add_issue(issues, "dummyjson_products_raw", "missing_values", "warning", cnt, f"Nulls in {col}", files["products_raw_json"])
    dups = duplicate_count(products_df, subset=["id"]) if "id" in products_df.columns else 0
    if dups > 0:
        add_issue(issues, "dummyjson_products_raw", "duplicates", "error", dups, "Duplicate product IDs", files["products_raw_json"])
    bad_price = invalid_numeric_range_count(products_df, "price", min_value=0)
    if bad_price and bad_price > 0:
        add_issue(issues, "dummyjson_products_raw", "range", "error", bad_price, "Negative prices", files["products_raw_json"])
    bad_rating = invalid_numeric_range_count(products_df, "rating", min_value=1, max_value=5)
    if bad_rating and bad_rating > 0:
        add_issue(issues, "dummyjson_products_raw", "range", "warning", bad_rating, "Ratings outside 1-5", files["products_raw_json"])
    bad_stock = invalid_numeric_range_count(products_df, "stock", min_value=0)
    if bad_stock and bad_stock > 0:
        add_issue(issues, "dummyjson_products_raw", "range", "error", bad_stock, "Negative stock", files["products_raw_json"])
    summaries.append(build_dataset_summary("dummyjson_products_raw", products_df, issues))

    mc = missing_columns(categories_df, ["category"])
    if mc:
        add_issue(issues, "dummyjson_categories_raw", "schema", "error", len(mc), f"Missing columns: {mc}", files["categories_raw_json"])
    for col, cnt in null_summary(categories_df).items():
        add_issue(issues, "dummyjson_categories_raw", "missing_values", "warning", cnt, f"Nulls in {col}", files["categories_raw_json"])
    dups = duplicate_count(categories_df, subset=["category"]) if "category" in categories_df.columns else 0
    if dups > 0:
        add_issue(issues, "dummyjson_categories_raw", "duplicates", "warning", dups, "Duplicate categories", files["categories_raw_json"])
    summaries.append(build_dataset_summary("dummyjson_categories_raw", categories_df, issues))

def validate_bronze(files, issues, summaries):
    for key in ["products_parquet", "categories_parquet"]:
        path = files.get(key)
        if path is None:
            continue
        df = pd.read_parquet(path)
        required_meta = ["source_system", "source_file", "batch_id", "ingestion_ts"]
        mc = missing_columns(df, required_meta)
        if mc:
            add_issue(issues, key, "schema", "warning", len(mc), f"Missing bronze metadata columns: {mc}", path)
        if len(df) == 0:
            add_issue(issues, key, "completeness", "error", 1, "Empty bronze file", path)
        summaries.append(build_dataset_summary(key, df, issues))

def main():
    log_event("validation", "started", "Validation started")

    issues = []
    summaries = []
    files = discover_files()

    print_discovered_files(files)
    validate_file_presence(files, issues)
    validate_retailrocket(files, issues, summaries)
    validate_dummyjson(files, issues, summaries)
    validate_bronze(files, issues, summaries)

    issues_df = pd.DataFrame(issues)
    summary_df = pd.DataFrame(summaries)

    if issues_df.empty:
        issues_df = pd.DataFrame(columns=["dataset", "check_type", "severity", "issue_count", "description", "file_path"])
    if summary_df.empty:
        summary_df = pd.DataFrame(columns=["dataset", "rows", "columns", "errors", "warnings", "status"])

    issues_df.to_csv(ISSUES_FILE, index=False)
    summary_df.to_csv(SUMMARY_FILE, index=False)

    report = {
        "run_id": RUN_ID,
        "run_ts": RUN_TS,
        "project_root": str(PROJECT_ROOT),
        "raw_root": str(RAW_ROOT),
        "discovered_files": {k: (str(v) if v else None) for k, v in files.items()},
        "overall_status": "PASS" if len(issues_df[issues_df["severity"] == "error"]) == 0 else "FAIL",
        "issues_file": str(ISSUES_FILE),
        "summary_file": str(SUMMARY_FILE),
        "datasets": summary_df.to_dict(orient="records"),
    }

    with open(REPORT_FILE, "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2)

    print("\nOverall status:", report["overall_status"])
    print("Issues file:", ISSUES_FILE)
    print("Summary file:", SUMMARY_FILE)
    print("Report file:", REPORT_FILE)

    return files, issues_df, summary_df, report

files, issues_df, summary_df, report = main()


PROJECT_ROOT: C:\Users\barath\recomart-pipeline
RAW_ROOT: C:\Users\barath\recomart-pipeline\data\raw
BRONZE_ROOT: C:\Users\barath\recomart-pipeline\data\bronze

Discovered files:
- events_csv: C:\Users\barath\recomart-pipeline\data\raw\retailrocket\load_date=2026-04-26\load_hour=15\events.csv
- category_tree_csv: C:\Users\barath\recomart-pipeline\data\raw\retailrocket\load_date=2026-04-26\load_hour=15\category_tree.csv
- item_properties_part1_csv: C:\Users\barath\recomart-pipeline\data\raw\retailrocket\load_date=2026-04-26\load_hour=15\item_properties_part1.csv
- item_properties_part2_csv: C:\Users\barath\recomart-pipeline\data\raw\retailrocket\load_date=2026-04-26\load_hour=15\item_properties_part2.csv
- products_raw_json: C:\Users\barath\recomart-pipeline\data\raw\dummyjson\load_date=2026-04-26\load_hour=15\products_raw.json
- categories_raw_json: C:\Users\barath\recomart-pipeline\data\raw\dummyjson\load_date=2026-04-26\load_hour=15\categories_raw.json
- products_parquet: C:\Users\ba